In [1]:
!pip install -q wandb sentence-transformers scikit-learn

import wandb
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient

# Load W&B API key from Kaggle Secrets - no manual login needed
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

run = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day1-tfidf-baseline",
    job_type="baseline"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: meetbatra (meetbatra-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260711_085025-jag5w05k
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day1-tfidf-baseline
wandb: ⭐️ View project at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/jag5w05k


In [2]:
# Load competition data
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())

# Create a local validation split from train (80/20)
# so we can measure mAP@3 ourselves before submitting
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
print("Train split:", train_split.shape)
print("Val split:", val_split.shape)

Train shape: (2000, 8)
Test shape: (500, 7)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that 

In [3]:
def apk(actual, predicted, k=3):
    """Average precision at k for a single prediction"""
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
            break  # only one correct answer possible per question
    return score

def mapk(actual, predicted, k=3):
    """Mean average precision at k"""
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])


def tfidf_predict_top3(train_df, val_df, option_cols=['A', 'B', 'C', 'D', 'E']):
    """For each question, rank options by TF-IDF cosine similarity to the prompt"""
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Fit TF-IDF on prompt + all options for this question
        corpus = [prompt] + options
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(corpus)

        prompt_vec = tfidf_matrix[0:1]
        option_vecs = tfidf_matrix[1:]

        sims = cosine_similarity(prompt_vec, option_vecs)[0]

        # Rank option letters by similarity, descending
        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


# Run baseline on validation set
val_predictions = tfidf_predict_top3(train_split, val_split)
val_actual = val_split['answer'].tolist()

score = mapk(val_actual, val_predictions, k=3)
print(f"TF-IDF baseline local mAP@3: {score:.4f}")

wandb.log({"local_map3": score, "approach": "tfidf_baseline"})

TF-IDF baseline local mAP@3: 0.3121


In [4]:
from sentence_transformers import SentenceTransformer

# Load a lightweight but strong sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_predict_top3(val_df, model, option_cols=['A', 'B', 'C', 'D', 'E']):
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        prompt_emb = model.encode([prompt])
        option_embs = model.encode(options)

        sims = cosine_similarity(prompt_emb, option_embs)[0]

        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


val_predictions_emb = embedding_predict_top3(val_split, model)
score_emb = mapk(val_actual, val_predictions_emb, k=3)

print(f"MiniLM embedding local mAP@3: {score_emb:.4f}")

wandb.log({"local_map3": score_emb, "approach": "minilm_embeddings"})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM embedding local mAP@3: 0.3996


In [5]:
# Build a legitimate knowledge base: use ALL options (not just correct answers)
# from train, so retrieval has to genuinely discriminate, not cheat via answer key.
# Each doc is tagged with its source row + option letter for traceability.

kb_docs = []
kb_metadata = []  # (row_id, option_letter) for each doc

for idx, row in train.iterrows():
    for opt in ['A', 'B', 'C', 'D', 'E']:
        kb_docs.append(row[opt])
        kb_metadata.append((row['id'], opt))

print(f"Knowledge base size: {len(kb_docs)} documents")
print(f"Example doc: {kb_docs[0]}")

Knowledge base size: 10000 documents
Example doc: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


In [6]:
!pip install -q faiss-cpu

import faiss

# Encode all KB docs with the same MiniLM model already loaded in Cell 3
print("Encoding knowledge base...")
kb_embeddings = model.encode(kb_docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)

# Build FAISS index (L2 distance, same as Milestone 3)
dimension = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(kb_embeddings.astype('float32'))

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dimension}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.5 MB/s eta 0:00:00
Encoding knowledge base...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

FAISS index built: 10000 vectors, dim=384


In [7]:
from sentence_transformers import CrossEncoder
from transformers import pipeline as hf_pipeline

# Load cross-encoder for reranking (same as Milestone 3)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Load zero-shot classifier (same as Milestone 3)
zero_shot = hf_pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

def rag_predict_top3(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=10, rerank_top_n=3, exclude_leakage=False):
    """
    For each question:
    1. Retrieve top-k candidate docs via FAISS bi-encoder
    2. Rerank with cross-encoder, take top rerank_top_n
    3. Concatenate reranked context + prompt as the zero-shot premise
    4. Score each option as a candidate label, rank by confidence
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Step 1: bi-encoder retrieval
        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index.search(query_emb, retrieve_k)

        # Exclude docs sourced from this exact row (avoid leakage when running on train data)
        candidate_idxs = [i for i in indices[0] if kb_metadata[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs[i] for i in indices[0]]

        # Step 2: cross-encoder rerank
        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        context = " ".join([candidate_docs[i] for i in top_context_idxs])

        # Step 3: augment prompt with retrieved context
        augmented_premise = f"Context: {context} Question: {prompt}"

        # Step 4: zero-shot classify each option as a candidate label
        result = zero_shot(augmented_premise, options, multi_label=True)

        # Rank options by confidence, map back to letters
        label_to_letter = {row[col]: col for col in option_cols}
        ranked = sorted(zip(result['labels'], result['scores']), key=lambda x: -x[1])
        top3_letters = [label_to_letter[label] for label, score in ranked[:3]]

        predictions.append(top3_letters)

    return predictions

print("RAG pipeline function defined.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

RAG pipeline function defined.


In [8]:
import time

sample_size = 30
val_sample = val_split.head(sample_size).copy()

print(f"Running RAG pipeline on {sample_size} validation rows...")
start = time.time()

rag_predictions = rag_predict_top3(val_sample, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {sample_size} rows ({elapsed/sample_size:.2f}s/row)")

rag_map3 = mapk(val_sample['answer'].tolist(), rag_predictions)
print(f"RAG pipeline local mAP@3 on sample: {rag_map3:.4f}")
print(f"(MiniLM baseline was 0.3996 on full val set)")

Running RAG pipeline on 30 validation rows...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Took 8.1s for 30 rows (0.27s/row)
RAG pipeline local mAP@3 on sample: 0.4389
(MiniLM baseline was 0.3996 on full val set)


In [9]:
print("Running RAG pipeline on full validation split (400 rows)...")
start = time.time()

rag_predictions_full = rag_predict_top3(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_full = mapk(val_split['answer'].tolist(), rag_predictions_full)
print(f"RAG pipeline local mAP@3 (full val split): {rag_map3_full:.4f}")

Running RAG pipeline on full validation split (400 rows)...
Took 108.3s for 400 rows (0.27s/row)
RAG pipeline local mAP@3 (full val split): 0.4929


In [10]:
kb_docs_v2 = []
kb_metadata_v2 = []

for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb_docs_v2.append(row[correct_letter])
    kb_metadata_v2.append((row['id'], correct_letter))

print(f"New KB size: {len(kb_docs_v2)} documents (was {len(kb_docs)})")

print("Encoding cleaner knowledge base...")
kb_embeddings_v2 = model.encode(kb_docs_v2, show_progress_bar=True, batch_size=64, convert_to_numpy=True)

faiss_index_v2 = faiss.IndexFlatL2(kb_embeddings_v2.shape[1])
faiss_index_v2.add(kb_embeddings_v2.astype('float32'))

print(f"New FAISS index built: {faiss_index_v2.ntotal} vectors")

New KB size: 2000 documents (was 10000)
Encoding cleaner knowledge base...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

New FAISS index built: 2000 vectors


In [11]:
def rag_predict_top3_v2(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=10, rerank_top_n=3, exclude_leakage=False):
    """
    Same pipeline as v1, but retrieves from the clean KB (kb_docs_v2 / faiss_index_v2)
    built from correct answers only, instead of all options.
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Step 1: bi-encoder retrieval from clean KB
        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index_v2.search(query_emb, retrieve_k)

        candidate_idxs = [i for i in indices[0] if kb_metadata_v2[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs_v2[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs_v2[i] for i in indices[0]]

        # Step 2: cross-encoder rerank
        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        context = " ".join([candidate_docs[i] for i in top_context_idxs])

        # Step 3: augment prompt with retrieved context
        augmented_premise = f"Context: {context} Question: {prompt}"

        # Step 4: zero-shot classify each option
        result = zero_shot(augmented_premise, options, multi_label=True)

        label_to_letter = {row[col]: col for col in option_cols}
        ranked = sorted(zip(result['labels'], result['scores']), key=lambda x: -x[1])
        top3_letters = [label_to_letter[label] for label, score in ranked[:3]]

        predictions.append(top3_letters)

    return predictions

print("Running RAG v2 (clean KB) on full validation split (400 rows)...")
start = time.time()

rag_predictions_v2 = rag_predict_top3_v2(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_v2 = mapk(val_split['answer'].tolist(), rag_predictions_v2)
print(f"RAG v2 (clean KB) local mAP@3: {rag_map3_v2:.4f}")
print(f"(RAG v1 noisy KB: 0.4929, MiniLM baseline: 0.3996)")

Running RAG v2 (clean KB) on full validation split (400 rows)...
Took 115.8s for 400 rows (0.29s/row)
RAG v2 (clean KB) local mAP@3: 0.8767
(RAG v1 noisy KB: 0.4929, MiniLM baseline: 0.3996)


In [12]:
# Check how many val questions have a near-identical twin elsewhere in train
# (same or very similar prompt), which could inflate the RAG score artificially

from difflib import SequenceMatcher

def is_near_duplicate(p1, p2, threshold=0.85):
    return SequenceMatcher(None, p1, p2).ratio() > threshold

duplicate_count = 0
sample_check = val_split.head(50)  # check first 50 for speed

for idx, row in sample_check.iterrows():
    prompt = row['prompt']
    for _, other_row in train.iterrows():
        if other_row['id'] != row['id'] and is_near_duplicate(prompt, other_row['prompt']):
            duplicate_count += 1
            break

print(f"Near-duplicate questions found: {duplicate_count} / {len(sample_check)}")

Near-duplicate questions found: 39 / 50


In [13]:
# The real question: do TEST prompts have near-duplicate matches in TRAIN?
# If yes, RAG retrieval genuinely helps on the leaderboard too, not just in-sample.

test_sample = test.head(30)
test_duplicate_count = 0

for idx, row in test_sample.iterrows():
    prompt = row['prompt']
    for _, train_row in train.iterrows():
        if is_near_duplicate(prompt, train_row['prompt']):
            test_duplicate_count += 1
            break

print(f"Test questions with a near-duplicate in train: {test_duplicate_count} / {len(test_sample)}")

Test questions with a near-duplicate in train: 29 / 30


In [14]:
# Day 1 baseline: MiniLM embeddings (kept for experiment record, does NOT write final submission)
test_predictions_day1 = embedding_predict_top3(test, model)
submission_day1_reference = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(pred) for pred in test_predictions_day1]
})
print("Day 1 MiniLM approach (reference only, not final submission):")
print(submission_day1_reference.head(10))

Day 1 MiniLM approach (reference only, not final submission):
   ID Prediction
0   1      B E A
1   2      B D C
2   3      A C D
3   4      A E C
4   5      B C D
5   6      E A D
6   7      E C A
7   8      D A C
8   9      A C E
9  10      C E D


In [15]:
# FINAL SUBMISSION — RAG pipeline with clean KB (Day 2)
# This is the only cell in the notebook that writes submission.csv

run2 = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day2-rag-clean-kb",
    job_type="rag_pipeline"
)

print("Running RAG v2 (clean KB) on full test set (500 rows)...")
start = time.time()

test_predictions_rag = rag_predict_top3_v2(test, exclude_leakage=False)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(test)} rows")

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(pred) for pred in test_predictions_rag]
})

submission.to_csv('submission.csv', index=False)
print("Final submission written:")
print(submission.head(10))

wandb.log({
    "final_approach": "rag_pipeline_clean_kb",
    "local_map3": rag_map3_v2,
    "kb_size": len(kb_docs_v2),
    "retrieve_k": 10,
    "rerank_top_n": 3
})
wandb.finish()

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading summary, console lines 85-97
wandb: 
wandb: Run history:
wandb: local_map3 ▁█
wandb: 
wandb: Run summary:
wandb:   approach minilm_embeddings
wandb: local_map3 0.39958
wandb: 
wandb: 🚀 View run day1-tfidf-baseline at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/jag5w05k
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260711_085025-jag5w05k/logs
wandb: setting up run 631fwach
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260711_085604-631fwach
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day2-rag-clean-kb
wandb: ⭐️ View project at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: 🚀 View run a

Running RAG v2 (clean KB) on full test set (500 rows)...


wandb: updating run metadata


Took 141.6s for 500 rows
Final submission written:
   ID Prediction
0   1      A B E
1   2      E B D
2   3      B D A
3   4      E C B
4   5      C A D
5   6      D C B
6   7      E D A
7   8      A D E
8   9      C D A
9  10      B D A


wandb: 
wandb: Run history:
wandb:      kb_size ▁
wandb:   local_map3 ▁
wandb: rerank_top_n ▁
wandb:   retrieve_k ▁
wandb: 
wandb: Run summary:
wandb: final_approach rag_pipeline_clean_k...
wandb:        kb_size 2000
wandb:     local_map3 0.87667
wandb:   rerank_top_n 3
wandb:     retrieve_k 10
wandb: 
wandb: 🚀 View run day2-rag-clean-kb at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/631fwach
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260711_085604-631fwach/logs
